# Phase 5 — NB1: Stage 1 Category Detection Training (R1 Retrain)

**Goal:** Retrain Stage 1 with fixed hyperparams: `encoder_lr=1e-5`, `pos_weight_cap=3.0`

**Fixes vs previous run:**
- V1: encoder_lr 2e-6 → **1e-5** (DeBERTa can actually fine-tune)
- V2: pos_weight cap 5.0 → **3.0** (less aggressive for rare categories)
- epochs 15 → **20** (more room to converge, patience=5 will stop early if needed)

**Input:** `lcminhc/semeval-absa-restaurant` (raw XML)

**Output:** Upload `/kaggle/working/outputs_p5_nb1/` as Kaggle dataset `p5-nb1-stage1`

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import json
import shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Wire SemEval XML data
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
assert KAGGLE_INPUT, 'Dataset semeval-absa-restaurant not found'
print(f'SemEval input: {KAGGLE_INPUT}')

os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training', exist_ok=True)
os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant', exist_ok=True)

shutil.copy(f'{KAGGLE_INPUT}/semeval16_restaurants_train.xml',
            'SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training/ABSA16_Restaurants_Train_SB1_v2.xml')
shutil.copy(f'{KAGGLE_INPUT}/semeval16_restaurants_test.xml',
            'SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant/EN_REST_SB1_TEST.xml.gold')
print('XML files in place.')

## 1. Prepare Data

In [ ]:
!python scripts/01_prepare_data.py

In [ ]:
# Verify new data files
for f in ['category_detection.jsonl', 'sentiment_records.jsonl']:
    path = f'data/processed/{f}'
    if os.path.exists(path):
        with open(path) as fp:
            n = sum(1 for _ in fp)
        print(f'{f}: {n} records')
    else:
        print(f'MISSING: {f}')

## 2. Train Stage 1 (Category Detection)

In [ ]:
import torch
import gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# R1: encoder_lr=1e-5, pos_weight_cap=3.0, epochs=20
!python scripts/04a_train_stage1.py --config configs/stage1_r1.yaml

## 3. Training Log

In [ ]:
log_path = 'logs/stage1_r1_training.jsonl'
if os.path.exists(log_path):
    print(f'{"Epoch":<8} {"Loss":<10} {"Cat F1":<10} {"Cat P":<10} {"Cat R":<10}')
    print('-' * 48)
    with open(log_path) as f:
        for line in f:
            rec = json.loads(line)
            print(f"{rec['epoch']:<8} {rec['train_loss']:<10.4f} {rec['category_f1']:<10.4f} "
                  f"{rec['category_precision']:<10.4f} {rec['category_recall']:<10.4f}")
else:
    print('No training log found.')

## 4. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb1'
os.makedirs(output_dir, exist_ok=True)

# Stage 1 checkpoint (model + thresholds)
ckpt_path = 'checkpoints/stage1_r1/best.pt'
if os.path.exists(ckpt_path):
    shutil.copy(ckpt_path, f'{output_dir}/stage1_best.pt')
    size_mb = os.path.getsize(ckpt_path) / 1e6
    print(f'stage1_best.pt: {size_mb:.1f} MB')
else:
    print(f'WARNING: {ckpt_path} not found!')

# New data files (needed by NB2 and NB3)
for f in ['category_detection.jsonl', 'sentiment_records.jsonl']:
    src = f'data/processed/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{f}')
        print(f'{f} copied')

# Logs
if os.path.exists('logs'):
    shutil.copytree('logs', f'{output_dir}/logs', dirs_exist_ok=True)
    print('logs/ copied')

print(f'\nOutputs saved to {output_dir}')
print('Upload as Kaggle dataset: p5-nb1-stage1')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_nb1_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_nb1')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_nb1_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')